# ARC-AGI-3 STaR LoRA — Colab QLoRA training (A100 40GB)

Runs the same train_lora.py as the Kaggle kernel but in QLORA mode
(4-bit NF4 base) so the 27B fits in 40GB. Expected: ~3-4h for
SUBSAMPLE=1200, 1 epoch.

**Runtime: pick A100 (high-RAM) via Runtime > Change runtime type.**
L4-24GB likely OOMs at seq 8192 (vocab 248k logits); if stuck on L4,
set MAX_LEN to 4096 below (halves logit memory).

You need your Kaggle `credentials.json` (the OAuth one from
`~/.kaggle/credentials.json`) — the notebook asks for it in cell 2.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       'peft==0.17.0', 'accelerate>=1.6', 'datasets>=3.0',
                       'transformers>=4.56.0', 'bitsandbytes>=0.45.0',
                       'kaggle==1.7.4.5'])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'pick a GPU runtime (A100 recommended)'

In [ ]:
import json, os, pathlib, subprocess
kd = pathlib.Path.home() / '.kaggle'; kd.mkdir(exist_ok=True)
kj = kd / 'kaggle.json'
kj.write_text(json.dumps({'username': 'takumuhata', 'key': 'REPLACE_WITH_KAGGLE_KEY'}))
os.chmod(kj, 0o600)
r = subprocess.run(['kaggle', 'datasets', 'list', '-m'], capture_output=True, text=True)
print((r.stdout or r.stderr)[:400])

In [ ]:
import os, pathlib, subprocess
BASE = pathlib.Path('/content/base_27b')
SFTD = pathlib.Path('/content/sft_data')
if not (BASE / 'config.json').exists():
    BASE.mkdir(exist_ok=True)
    print('downloading bf16 base (~44GB, ~20-60min)...')
    subprocess.check_call(['kaggle', 'datasets', 'download',
                           'rahim3/qwen3-8-27b-bf16', '--unzip',
                           '-p', str(BASE)])
if not (SFTD / 'packed_sft.jsonl').exists():
    SFTD.mkdir(exist_ok=True)
    subprocess.check_call(['kaggle', 'datasets', 'download',
                           'takumuhata/taaf-duck-sft-v1', '--unzip',
                           '-p', str(SFTD)])
print('base files:', len(list(BASE.glob('*.safetensors'))), 'shards')
print('sft files:', sorted(p.name for p in SFTD.iterdir()))

In [ ]:
import json, pathlib, subprocess
ad = pathlib.Path('/content/lora_adapter')
assert (ad / 'adapter_config.json').exists() and (ad / 'adapter_model.safetensors').exists(), 'adapter incomplete'
(ad / 'dataset-metadata.json').write_text(json.dumps({
    'title': 'taaf-duck-lora-v1', 'id': 'takumuhata/taaf-duck-lora-v1',
    'licenses': [{'name': 'other'}]}))
r = subprocess.run(['kaggle', 'datasets', 'version', '-p', str(ad),
                    '-m', 'qlora adapter (colab a100)'], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
if r.returncode != 0:
    r = subprocess.run(['kaggle', 'datasets', 'create', '-p', str(ad)],
                       capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-500:])
    assert r.returncode == 0, 'dataset upload failed'
print('uploaded -> takumuhata/taaf-duck-lora-v1')
print('next: push submit_ag3_lora on kaggle (still needs ~7h kaggle GPU)')

In [ ]:
import json, pathlib, subprocess
ad = pathlib.Path('/content/lora_adapter')
assert (ad / 'adapter_config.json').exists() and \
       (ad / 'adapter_model.safetensors').exists(), 'adapter incomplete'
(ad / 'dataset-metadata.json').write_text(json.dumps({
    'title': 'taaf-duck-lora-v1',
    'id': 'takumuhata/taaf-duck-lora-v1',
    'licenses': [{'name': 'other'}],
}))
r = subprocess.run(['kaggle', 'datasets', 'status',
                    'takumuhata/taaf-duck-lora-v1'],
                   capture_output=True, text=True)
if r.returncode == 0:
    subprocess.check_call(['kaggle', 'datasets', 'version', '-p', str(ad),
                           '-m', 'qlora adapter (colab a100)'])
else:
    subprocess.check_call(['kaggle', 'datasets', 'create', '-p', str(ad)])
print('uploaded -> takumuhata/taaf-duck-lora-v1')
print('next: on the Devin side run arc-agi-3/sat_lora_pipeline.sh step 3'
      ' (or just push submit_ag3_lora) to eval the adapter')